# h5ad column extractor

Uses `h5ad_extractor` (backed read) to pull `obs` or `var` columns into Parquet or CSV with the AnnData row index preserved.

In [1]:
import pandas as pd
import scanpy as sc

from h5ad_extractor import H5adExtractConfig, extract_annotation_columns
from shared.repo import REPO_ROOT

ROOT = REPO_ROOT

## Paths and input file

`ROOT` is [`REPO_ROOT`](../scripts/shared/repo.py) (repo root from `.git` / `.venv` walk). Adjust `H5AD_PATH` if needed. By default we use `tmp/` when present, otherwise the first raw `.h5ad` under the scBaseCount data directory, otherwise a clustered file under `output/cytetype/data/` if present.

In [2]:
H5AD_DIR = ROOT / "data/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens"

H5AD_PATH = ROOT / "output/cytetype/data/SRX17412841_cytetype_annotated.h5ad"

print(H5AD_PATH)

/Users/otodreas/Desktop/Work/Nygen/scBaseCount_Pipeline/output/cytetype/data/SRX17412841_cytetype_annotated.h5ad


## Preview available columns

In [3]:
adata = sc.read(str(H5AD_PATH), backed="r")
try:
    print("obs columns:", list(adata.obs.columns))
    print("n cells", adata.n_obs, " n genes", adata.n_vars)
finally:
    if getattr(adata, "isbacked", False) and adata.file is not None:
        adata.file.close()

obs columns: ['gene_count_Unique', 'umi_count_Unique', 'gene_count_UniqueAndMult-EM', 'umi_count_UniqueAndMult-EM', 'gene_count_UniqueAndMult-Uniform', 'umi_count_UniqueAndMult-Uniform', 'SRX_accession', 'cell_type', 'cell_ontology_term_id', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0', 'leiden_1.1', 'leiden_1.2', 'leiden_1.3', 'leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_merged', 'cytetype_annotation_leiden_merged', 'cytetype_cellOntologyTerm_leiden_merged', 'cytetype_cellOntologyTermID_leiden_merged', 'cytetype_cellState_leiden_merged']
n cells 4757  n genes 24280


## Extract and save

Set `COLUMN_NAMES` to columns that exist in `adata.obs` (or set `annotationAxis="var"` and use `adata.var` names). By default, files go to `output/h5ad_extract/` under the repo root (`H5adExtractConfig.outputDir`), as `{h5ad_stem}_{obs|var}_columns.parquet` or `.csv`. Optional `outputPath` overrides that with a specific file path (still resolved with `REPO_ROOT` when relative). `gs://` h5ad paths are supported.

In [4]:
COLUMN_NAMES = ["cell_type", "leiden_merged", "cytetype_annotation_leiden_merged"]

pq_path = extract_annotation_columns(
    H5adExtractConfig(
        h5adPath=H5AD_PATH,
        columnNames=COLUMN_NAMES,
        outputFormat="parquet",
    )
)
csv_path = extract_annotation_columns(
    H5adExtractConfig(
        h5adPath=H5AD_PATH,
        columnNames=COLUMN_NAMES,
        outputFormat="csv",
    )
)

print(pq_path)
print(csv_path)

/Users/otodreas/Desktop/Work/Nygen/scBaseCount_Pipeline/output/h5ad_extract/SRX17412841_cytetype_annotated_obs_columns.parquet
/Users/otodreas/Desktop/Work/Nygen/scBaseCount_Pipeline/output/h5ad_extract/SRX17412841_cytetype_annotated_obs_columns.csv


## Load outputs

In [20]:
df_pq = pd.read_parquet(pq_path)
df_csv = pd.read_csv(csv_path)

df_pq["leiden_merged"] = df_pq["leiden_merged"].astype(int)
df_csv["leiden_merged"] = df_csv["leiden_merged"].astype(int)

# display(df_pq.head())
# print(f"df_pq.shape == df_csv.shape: {df_pq.shape == df_csv.shape}")
# df_pq.shape

# for df in [df_pq, df_csv]:
#     print(df.columns)
#     print(df.shape)
#     print(df.head())

# df_pq["leiden_merged"].astype(int) == df_csv["leiden_merged"].astype(int)

(df_pq == df_csv).value_counts()

cell_type  leiden_merged  cytetype_annotation_leiden_merged
True       True           True                                 4757
Name: count, dtype: int64